In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold

# ==========================================
# 1. Load data
# ==========================================
print("Loading data...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# ==========================================
# 2. Preprocessing function
# ==========================================
def preprocess(df, is_train=True, impute_stats=None):
    df = df.copy()
    
    # Extract hour and minute
    df[['hour', 'minute']] = df['timestamp'].str.split(':', expand=True).astype(int)
    df = df.drop(columns=['timestamp'])
    
    # Cyclic time features
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
    df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60)
    df = df.drop(columns=['hour', 'minute'])
    
    # Fill Missing Values
    if is_train:
        road_type_mode = df['RoadType'].mode()[0]
        weather_mode = df['Weather'].mode()[0]
        temp_median = df['Temperature'].median()
        
        df['RoadType'] = df['RoadType'].fillna(road_type_mode)
        df['Weather'] = df['Weather'].fillna(weather_mode)
        df['Temperature'] = df['Temperature'].fillna(temp_median)
        
        impute_stats = {
            'road_type_mode': road_type_mode,
            'weather_mode': weather_mode,
            'temp_median': temp_median
        }
        return df, impute_stats
    else:
        df['RoadType'] = df['RoadType'].fillna(impute_stats['road_type_mode'])
        df['Weather'] = df['Weather'].fillna(impute_stats['weather_mode'])
        df['Temperature'] = df['Temperature'].fillna(impute_stats['temp_median'])
        return df

print("Preprocessing data...")
train_proc, impute_stats = preprocess(train, is_train=True)
test_proc = preprocess(test, is_train=False, impute_stats=impute_stats)

# ==========================================
# 3. Categorical Encoding (Track and Save)
# ==========================================
print("Encoding categorical features...")
cat_cols = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    le.fit(list(train_proc[col].astype(str)) + list(test_proc[col].astype(str)))
    train_proc[col] = le.transform(train_proc[col].astype(str))
    test_proc[col] = le.transform(test_proc[col].astype(str))
    encoders[col] = le  # Keep track of individual encoders for the api

# ==========================================
# 4. Aggregation Features (MAXIMIZED)
# ==========================================
print("Adding advanced aggregation features...")

# 4a. Geohash mean
geohash_mean = train_proc.groupby('geohash')['demand'].mean()
train_proc['geohash_demand_mean'] = train_proc['geohash'].map(geohash_mean).fillna(0)
test_proc['geohash_demand_mean'] = test_proc['geohash'].map(geohash_mean).fillna(0)

# 4b. Geohash × Hour mean (The Golden Feature)
train_proc['hour_temp'] = train['timestamp'].str.split(':').str[0].astype(int)
test_proc['hour_temp'] = test['timestamp'].str.split(':').str[0].astype(int)

geohash_hour_mean = train_proc.groupby(['geohash', 'hour_temp'])['demand'].mean()
train_proc['geohash_hour_demand_mean'] = train_proc.groupby(['geohash', 'hour_temp'])['demand'].transform('mean')

test_proc['geohash_hour_demand_mean'] = test_proc.merge(
    geohash_hour_mean.reset_index(), on=['geohash', 'hour_temp'], how='left', suffixes=('', '_test')
)['demand'].fillna(0)

train_proc = train_proc.drop(columns=['hour_temp'])
test_proc = test_proc.drop(columns=['hour_temp'])

# 4c. Day × Hour mean (Quick Win #1)
train_proc['day_hour_key'] = train_proc['day'].astype(str) + '_' + train_proc['hour_sin'].round(2).astype(str)
test_proc['day_hour_key'] = test_proc['day'].astype(str) + '_' + test_proc['hour_sin'].round(2).astype(str)

day_hour_mean = train_proc.groupby('day_hour_key')['demand'].mean()
train_proc['day_hour_demand_mean'] = train_proc['day_hour_key'].map(day_hour_mean).fillna(0)
test_proc['day_hour_demand_mean'] = test_proc['day_hour_key'].map(day_hour_mean).fillna(0)

train_proc = train_proc.drop(columns=['day_hour_key'])
test_proc = test_proc.drop(columns=['day_hour_key'])

# 4d. RoadType mean (Quick Win #2)
roadtype_mean = train_proc.groupby('RoadType')['demand'].mean()
train_proc['roadtype_demand_mean'] = train_proc['RoadType'].map(roadtype_mean).fillna(0)
test_proc['roadtype_demand_mean'] = test_proc['RoadType'].map(roadtype_mean).fillna(0)

# ==========================================
# 5. Prepare Train/Test sets
# ==========================================
X_train = train_proc.drop(columns=['Index', 'demand'])
y_train = train_proc['demand']
X_test = test_proc.drop(columns=['Index'])

# ==========================================
# 6. Train HistGradientBoostingRegressor Model (1000 Iterations)
# ==========================================
print("\nTraining HistGradientBoostingRegressor model with 1000 iterations...")
model = HistGradientBoostingRegressor(
    random_state=42, 
    max_iter=1000,
    learning_rate=0.05,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    verbose=0
)

# 5-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='r2')

print(f"\n✅ 5-Fold CV R² Scores: {cv_scores}")
print(f"Mean CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Estimated Leaderboard Score: {max(0, 100 * cv_scores.mean()):.2f}")

# Train final model
model.fit(X_train, y_train)

# ==========================================
# 7. Predict & Save Submission CSV
# ==========================================
print("\nPredicting on test set...")
preds = model.predict(X_test)

sub = pd.DataFrame({'Index': test['Index'], 'demand': preds})
sub.to_csv('baseline_submission.csv', index=False)
print("\n✅ DONE! Saved to 'baseline_submission.csv'")

# ==========================================
# 8. Export Components to Backend Folder
# ==========================================
print("\nExporting fresh model components for backend compatibility...")

# Save the modern scikit-learn model
joblib.dump(model, 'gridlock-ml/backend/model.pkl')

# Save the full label encoders dictionary
joblib.dump(encoders, 'gridlock-ml/backend/label_encoders.pkl')

# Save missing-value imputation values as JSON
with open('gridlock-ml/backend/impute_stats.json', 'w') as f:
    json.dump(impute_stats, f, indent=4)

# Save historical reference datasets for your API engine
geohash_mean.to_csv('gridlock-ml/backend/geohash_mean.csv')
geohash_hour_mean.to_csv('gridlock-ml/backend/geohash_hour_mean.csv')
roadtype_mean.to_csv('gridlock-ml/backend/roadtype_mean.csv')

# Day hour mean csv for any dynamic API calculations
day_hour_mean.to_csv('gridlock-ml/backend/day_hour_mean.csv')

print("✅ Backend sync complete! Run 'python api.py' now.")

Loading data...
Preprocessing data...
Adding advanced aggregation features...

Training HistGradientBoostingRegressor model with 1000 iterations...

✅ 5-Fold CV R² Scores: [0.9735718  0.97346163 0.97387783 0.97139214 0.97522794]
Mean CV R²: 0.9735 ± 0.0012
Estimated Leaderboard Score: 97.35

Predicting on test set...

✅ DONE! Saved to 'baseline_submission.csv'
